# Module 5.1: Deploy the Booking Agent to AgentCore Runtime

This notebook packages the Module 3.1 booking agent in a container and deploys it to :link[Amazon Bedrock AgentCore Runtime]{href="https://aws.amazon.com/bedrock/agentcore/" external=true}.

**Overview:**

- **AgentCore Runtime:** The managed service that runs the agent container.
- **Execution role:** The IAM role that lets the Runtime pull the image, write telemetry, and invoke Bedrock.
- **Smoke tests:** Five checks that confirm retrieval, safe responses, and reservation writes after deployment.

The Module 3.1 agent runs in your JupyterLab notebook kernel. That environment stores the Neo4j password, and its AWS credentials authorize Bedrock calls. This deployment changes where the agent runs and how callers invoke it:

| Module 3.1 | Module 5.1 |
|---|---|
| Runs in your kernel | Runs in a container AgentCore starts |
| The notebook environment holds the Neo4j password | The Runtime holds it, injected at launch |
| Reachable only from Jupyter | Invoked through `InvokeAgentRuntime` by authorized AWS clients |
| Session is your kernel's memory | Each invocation uses a caller-provided session ID |

The deployed agent reuses the retrieval code, grounding instructions, and reservation command. It exposes the reservation command as a second agent tool. It adds Runtime request handling and structured tool results. Both tools connect directly to Neo4j. Neo4j enforces the reservation rule in the same transaction that writes the reservation.

:::alert{type="warning" header="AWS resources created"}
This notebook creates one IAM execution role, one ECR repository, one CodeBuild project, and one AgentCore Runtime. The build takes three to five minutes. The workshop does not delete these resources automatically. Remove them when you finish.
:::

In [ ]:
import os
import sys
from pathlib import Path


def locate_notebooks_root():
    override = os.environ.get("WORKSHOP_NOTEBOOKS_DIR")
    if override:
        candidate = Path(override).expanduser().resolve()
        if (candidate / "workshop").is_dir():
            return candidate
        raise RuntimeError(
            "WORKSHOP_NOTEBOOKS_DIR must contain the workshop package"
        )

    start = Path.cwd().resolve()
    for candidate in (start, start / "notebooks", start.parent):
        if (candidate / "workshop").is_dir():
            return candidate
    raise RuntimeError(
        "Run from the repository root, notebooks/, or this module "
        "directory; or set WORKSHOP_NOTEBOOKS_DIR."
    )


NOTEBOOKS_ROOT = locate_notebooks_root()
REPO_ROOT = NOTEBOOKS_ROOT.parent
MODULE_DIR = NOTEBOOKS_ROOT / "05-agentcore-deploy"
if str(NOTEBOOKS_ROOT) not in sys.path:
    sys.path.insert(0, str(NOTEBOOKS_ROOT))
print(f"Workshop root: {REPO_ROOT}")

## 1. Check the values required for deployment

At launch, the notebook passes four Neo4j connection values to the Runtime as environment variables. This step checks those values and your AWS credentials before it creates AWS resources.

In [ ]:
import json
import shutil
import subprocess
import uuid
from datetime import date, timedelta
from pathlib import Path

import boto3
from dotenv import load_dotenv

from workshop.aws_region import configure_aws_region
from workshop.bedrock_providers import default_model_id

load_dotenv(NOTEBOOKS_ROOT / ".env")
load_dotenv(REPO_ROOT / ".env")
load_dotenv(REPO_ROOT / "CONFIG.txt")

REGION = configure_aws_region()
MODEL_ID = default_model_id()

# Every name this deploy creates, derived from one prefix. The starter toolkit
# derives the ECR repository and CodeBuild project names from the Runtime name,
# so cleanup can only find what this notebook created if it derives them the
# same way. AgentCore Runtime names accept letters, digits and underscores
# only, which is why this one carries no hyphen.
RUNTIME_NAME = "GraphRagBookingAgent"
ROLE_NAME = "workshop-graphrag-runtime-role"
ECR_REPO = "workshop-graphrag-booking-agent"
CB_PROJECT = f"bedrock-agentcore-{RUNTIME_NAME.lower()}-builder"

# The teardown gate. Cleanup deletes a resource only if it carries this exact
# key and value, and each of the three services below demands its own shape.
WORKSHOP_TAG_KEY = "WorkshopResource"
WORKSHOP_TAG_VALUE = "graphrag-with-neo4j"
WORKSHOP_TAGS_MAP = {WORKSHOP_TAG_KEY: WORKSHOP_TAG_VALUE}                          # agentcore
WORKSHOP_TAGS_KV = [{"Key": WORKSHOP_TAG_KEY, "Value": WORKSHOP_TAG_VALUE}]         # ecr, iam
WORKSHOP_TAGS_KV_LOWER = [{"key": WORKSHOP_TAG_KEY, "value": WORKSHOP_TAG_VALUE}]   # codebuild

# The deployed Runtime reads its Neo4j connection from these, so they have to be
# forwarded as container environment variables at launch.
NEO4J_ENV = ("NEO4J_URI", "NEO4J_USERNAME", "NEO4J_PASSWORD", "NEO4J_DATABASE")
NEO4J_VALUES = {name: os.getenv(name, "").strip() for name in NEO4J_ENV}

AWS_READY = boto3.Session().get_credentials() is not None
missing = [name for name, value in NEO4J_VALUES.items() if not value]
DEPLOY_READY = AWS_READY and not missing

print(f"region:          {REGION}")
print(f"model:           {MODEL_ID}")
print(f"runtime name:    {RUNTIME_NAME}")
print(f"execution role:  {ROLE_NAME}")
print(f"AWS credentials: {'found' if AWS_READY else 'NOT FOUND'}")
print(f"Neo4j values:    {'all four present' if not missing else 'missing ' + ', '.join(missing)}")

if not DEPLOY_READY:
    print("\nNot ready to deploy. Every live cell below will skip.")
    if missing:
        print(f"  Add to {REPO_ROOT / 'CONFIG.txt'}: {', '.join(missing)}")
    if not AWS_READY:
        print("  Configure AWS credentials before re-running this cell.")
else:
    print("\nReady to deploy.")

## 2. Put the shared code into the Docker build context

Docker copies files only from its build context. The agent needs two files outside that context: the shared `workshop/` package and Module 3.1's graph-enforced write path, `reservation_command.py`.

This step builds `workshop/` as a wheel and copies both dependencies into `runtime_app/`. The staged files are gitignored and replaced on every run. The original files remain the source of truth for each build.

In [ ]:
DEPLOY_DIR = MODULE_DIR / "runtime_app"
if not DEPLOY_DIR.is_dir():
    raise FileNotFoundError(
        "runtime_app/ not found. Run this notebook from 05-agentcore-deploy/."
    )

PACKAGE_SRC = NOTEBOOKS_ROOT / "workshop"
COMMAND_SRC = NOTEBOOKS_ROOT / "03-grounded-booking-agent" / "reservation_command.py"

# Staged unconditionally, even when the deploy will skip. It writes only inside
# runtime_app/, it is cheap, and it means a participant without AWS
# credentials can still read exactly what would have gone into the image.
staged_command = DEPLOY_DIR / "reservation_command.py"

# Old wheels removed first. `uv build` overwrites a wheel of the same name,
# but a stale one from a since-reverted version bump would otherwise sit
# next to the fresh one and match the Dockerfile's `workshop-*.whl` glob
# twice.
for old_wheel in DEPLOY_DIR.glob("workshop-*.whl"):
    old_wheel.unlink()
subprocess.run(
    ["uv", "build", "--wheel", "--out-dir", str(DEPLOY_DIR), str(PACKAGE_SRC)],
    check=True,
)
staged_wheel = next(DEPLOY_DIR.glob("workshop-*.whl"))
shutil.copy2(COMMAND_SRC, staged_command)

# Recorded into the build context itself, not just printed here. A printed
# commit lives in notebook output, which does not survive into the image;
# a file inside the build context does, so a rebuilt image can be checked
# against its own claim of what it was built from.
git_commit = subprocess.run(
    ["git", "rev-parse", "HEAD"], cwd=REPO_ROOT, capture_output=True, text=True, check=True
).stdout.strip()
git_dirty = bool(
    subprocess.run(
        ["git", "status", "--porcelain"], cwd=REPO_ROOT, capture_output=True, text=True, check=True
    ).stdout.strip()
)
(DEPLOY_DIR / "BUILD_INFO.txt").write_text(f"commit={git_commit}\ndirty={git_dirty}\n")

print(f"\nBuild commit: {git_commit}{'  (DIRTY TREE)' if git_dirty else ''}")
if git_dirty:
    print(
        "WARNING: uncommitted changes are present. This image will not trace to a\n"
        "git ref. Commit before deploying if this build needs to be reproducible."
    )

print(f"Staged {staged_wheel.relative_to(MODULE_DIR)}")
print(f"Staged {staged_command.relative_to(MODULE_DIR)}")
print("\nBuild context now contains:")
for path in sorted(DEPLOY_DIR.iterdir()):
    marker = "/" if path.is_dir() else ""
    print(f"  {path.name}{marker}")

## 3. Create the IAM role for the Runtime

AgentCore assumes this IAM role to run the container. The role lets the Runtime pull its image, write logs and traces, obtain workload identity tokens, and invoke Bedrock models. Neo4j access uses the credentials passed as environment variables at launch.

The Bedrock permissions include `foundation-model/*` and the account-scoped inference profile ARN. The workshop uses a cross-region inference profile that can route requests to several foundation-model ARNs. A policy limited to one model ARN would block requests routed to the other models.

In [ ]:
ROLE_ARN = ""

if not DEPLOY_READY:
    print("Skipping role creation: see Step 1.")
else:
    iam = boto3.client("iam")
    account_id = boto3.client("sts", region_name=REGION).get_caller_identity()["Account"]
    runtime_logs = f"arn:aws:logs:{REGION}:{account_id}:log-group:/aws/bedrock-agentcore/runtimes/"

    trust_policy = {
        "Version": "2012-10-17",
        "Statement": [
            {
                "Sid": "AssumeRolePolicy",
                "Effect": "Allow",
                "Principal": {"Service": "bedrock-agentcore.amazonaws.com"},
                "Action": "sts:AssumeRole",
                # Scoped to this account and this service. Without the
                # condition the role is assumable by AgentCore in any account
                # that learns its ARN, which is the confused-deputy shape the
                # service documentation warns about.
                "Condition": {
                    "StringEquals": {"aws:SourceAccount": account_id},
                    "ArnLike": {
                        "aws:SourceArn": f"arn:aws:bedrock-agentcore:{REGION}:{account_id}:*"
                    },
                },
            }
        ],
    }

    inline_policy = {
        "Version": "2012-10-17",
        "Statement": [
            {
                "Sid": "ECRImageAccess",
                "Effect": "Allow",
                "Action": ["ecr:BatchGetImage", "ecr:GetDownloadUrlForLayer"],
                "Resource": f"arn:aws:ecr:{REGION}:{account_id}:repository/*",
            },
            {
                "Sid": "ECRTokenAccess",
                "Effect": "Allow",
                "Action": ["ecr:GetAuthorizationToken"],
                "Resource": "*",
            },
            {
                "Sid": "RuntimeLogs",
                "Effect": "Allow",
                "Action": [
                    "logs:CreateLogGroup",
                    "logs:CreateLogStream",
                    "logs:PutLogEvents",
                    "logs:DescribeLogStreams",
                ],
                "Resource": f"{runtime_logs}*",
            },
            {
                "Sid": "DescribeLogGroups",
                "Effect": "Allow",
                "Action": ["logs:DescribeLogGroups"],
                "Resource": f"arn:aws:logs:{REGION}:{account_id}:log-group:*",
            },
            {
                "Sid": "XRay",
                "Effect": "Allow",
                "Action": [
                    "xray:PutTraceSegments",
                    "xray:PutTelemetryRecords",
                    "xray:GetSamplingRules",
                    "xray:GetSamplingTargets",
                ],
                "Resource": "*",
            },
            {
                "Sid": "CloudWatchMetrics",
                "Effect": "Allow",
                "Action": "cloudwatch:PutMetricData",
                "Resource": "*",
                "Condition": {
                    "StringEquals": {"cloudwatch:namespace": "bedrock-agentcore"}
                },
            },
            {
                "Sid": "GetAgentAccessToken",
                "Effect": "Allow",
                "Action": [
                    "bedrock-agentcore:GetWorkloadAccessToken",
                    "bedrock-agentcore:GetWorkloadAccessTokenForJWT",
                    "bedrock-agentcore:GetWorkloadAccessTokenForUserId",
                ],
                "Resource": [
                    f"arn:aws:bedrock-agentcore:{REGION}:{account_id}:workload-identity-directory/default",
                    f"arn:aws:bedrock-agentcore:{REGION}:{account_id}:workload-identity-directory/default/workload-identity/*",
                ],
            },
            {
                "Sid": "BedrockModelInvocation",
                "Effect": "Allow",
                "Action": ["bedrock:InvokeModel", "bedrock:InvokeModelWithResponseStream"],
                "Resource": [
                    "arn:aws:bedrock:*::foundation-model/*",
                    f"arn:aws:bedrock:{REGION}:{account_id}:inference-profile/*",
                ],
            },
        ],
    }

    # create_role then fall back to update. A blind delete-and-recreate would
    # break any Runtime already using the role, and re-running this notebook
    # should be safe.
    try:
        iam.create_role(
            RoleName=ROLE_NAME,
            AssumeRolePolicyDocument=json.dumps(trust_policy),
            Description="AgentCore Runtime execution role for the GraphRAG workshop",
            Tags=WORKSHOP_TAGS_KV,
        )
        print(f"Created role: {ROLE_NAME}")
    except iam.exceptions.EntityAlreadyExistsException:
        iam.update_assume_role_policy(
            RoleName=ROLE_NAME, PolicyDocument=json.dumps(trust_policy)
        )
        print(f"Role already exists, trust policy refreshed: {ROLE_NAME}")

    iam.put_role_policy(
        RoleName=ROLE_NAME,
        PolicyName="graphrag-runtime-policy",
        PolicyDocument=json.dumps(inline_policy),
    )
    ROLE_ARN = iam.get_role(RoleName=ROLE_NAME)["Role"]["Arn"]
    print(f"Execution role: {ROLE_ARN}")

## 4. Build the container and start the Runtime

The starter toolkit uses CodeBuild to build the image, pushes it to the ECR repository, and creates the Runtime. It uses the current directory as the build root. This cell changes to `runtime_app/` before launch so the toolkit uses the provided `Dockerfile` and sends only the intended build context.

The launch usually takes three to five minutes.

In [ ]:
RUNTIME_ARN = ""
RUNTIME_ID = None

if not DEPLOY_READY:
    print("Skipping launch: see Step 1.")
else:
    from bedrock_agentcore_starter_toolkit import Runtime

    if Path.cwd() != DEPLOY_DIR:
        os.chdir(DEPLOY_DIR)
    print(f"Build context: {Path.cwd()}")

    # Remove only the config file this notebook's toolkit run writes, in this
    # directory. It records the runtime ID of a previous deploy, and a stale one
    # sends the launch at a Runtime that may no longer exist.
    local_config = Path.cwd() / ".bedrock_agentcore.yaml"
    if local_config.exists():
        local_config.unlink()
        print(f"Removed stale toolkit config: {local_config.name}")

    # The participant IAM policy only grants ECR actions on repositories named
    # workshop-*, so the repository is created here under a name that policy
    # allows. Left to auto_create_ecr, the toolkit derives the name from the
    # runtime and produces one the policy denies, which fails the push for a
    # participant and succeeds only for an administrator.
    ecr_setup = boto3.client("ecr", region_name=REGION)
    try:
        repo = ecr_setup.create_repository(repositoryName=ECR_REPO)["repository"]
        print(f"Created ECR repository: {ECR_REPO}")
    except ecr_setup.exceptions.RepositoryAlreadyExistsException:
        repo = ecr_setup.describe_repositories(repositoryNames=[ECR_REPO])["repositories"][0]
        print(f"Reusing ECR repository: {ECR_REPO}")
    ECR_URI = repo["repositoryUri"]
    print(f"ECR URI:       {ECR_URI}")

    agent_runtime = Runtime()
    agent_runtime.configure(
        entrypoint="booking_agent.py",
        execution_role=ROLE_ARN,
        ecr_repository=ECR_URI,
        auto_create_ecr=False,
        requirements_file="agent_requirements.txt",
        region=REGION,
        agent_name=RUNTIME_NAME,
        deployment_type="container",
        non_interactive=True,
    )

    print("\nLaunching agent (3-5 minutes)...")
    result = agent_runtime.launch(
        auto_update_on_conflict=True,
        env_vars={
            # Both spellings. botocore reads only AWS_DEFAULT_REGION, and the
            # workshop documents AWS_REGION.
            "AWS_REGION": REGION,
            "AWS_DEFAULT_REGION": REGION,
            "MODEL_ID": MODEL_ID,
            **NEO4J_VALUES,
        },
    )

    RUNTIME_ARN = result.agent_arn
    if not RUNTIME_ARN:
        raise RuntimeError("launch() returned no agent ARN; read the CodeBuild logs.")

    # The trailing ARN segment is the runtime ID, and it names the CloudWatch
    # log group. Printing it saves a console hunt when a smoke test below
    # produces something worth reading the logs for.
    RUNTIME_ID = RUNTIME_ARN.split("/")[-1]
    os.chdir(DEPLOY_DIR.parent)

    print(f"\nAgent deployed: {RUNTIME_ARN}")
    print(f"Runtime ID:     {RUNTIME_ID}")
    print(f"Log group:      /aws/bedrock-agentcore/runtimes/{RUNTIME_ID}-DEFAULT")

## 5. Tag every resource created by the deployment

The previous step creates the ECR repository. The toolkit then creates the CodeBuild project and Runtime. The toolkit does not apply the workshop tag to those resources. Cleanup finds resources by that tag, so this step tags each resource and verifies the Runtime tag.

The code identifies each resource by its exact name or ARN.

In [ ]:
if not DEPLOY_READY:
    print("Skipping tagging: nothing was deployed.")
else:
    ecr_client = boto3.client("ecr", region_name=REGION)
    codebuild_client = boto3.client("codebuild", region_name=REGION)
    agentcore = boto3.client("bedrock-agentcore-control", region_name=REGION)

    try:
        repo = ecr_client.describe_repositories(repositoryNames=[ECR_REPO])["repositories"][0]
        ecr_client.tag_resource(resourceArn=repo["repositoryArn"], tags=WORKSHOP_TAGS_KV)
        print(f"Tagged ECR repository:  {ECR_REPO}")
    except ecr_client.exceptions.RepositoryNotFoundException:
        print(f"ECR repository not found (nothing to tag): {ECR_REPO}")

    # update_project replaces the whole tag set, so merge rather than clobber
    # whatever the toolkit put there.
    projects = codebuild_client.batch_get_projects(names=[CB_PROJECT])["projects"]
    if projects:
        merged = [t for t in projects[0].get("tags", []) if t.get("key") != WORKSHOP_TAG_KEY]
        codebuild_client.update_project(name=CB_PROJECT, tags=merged + WORKSHOP_TAGS_KV_LOWER)
        print(f"Tagged CodeBuild project: {CB_PROJECT}")
    else:
        print(f"CodeBuild project not found (nothing to tag): {CB_PROJECT}")

    if not RUNTIME_ARN:
        raise RuntimeError("RUNTIME_ARN is not set. Re-run the launch cell before tagging.")
    agentcore.tag_resource(resourceArn=RUNTIME_ARN, tags=WORKSHOP_TAGS_MAP)

    # Read the tags back rather than trusting the call. A tag that did not stick
    # is a resource teardown will silently leave running.
    runtime_tags = agentcore.list_tags_for_resource(resourceArn=RUNTIME_ARN).get("tags", {})
    if runtime_tags.get(WORKSHOP_TAG_KEY) != WORKSHOP_TAG_VALUE:
        raise RuntimeError(f"Runtime tag did not stick. Read back: {runtime_tags}")
    print(f"Tagged AgentCore Runtime: {RUNTIME_ARN}")
    print("\nAll toolkit-created resources tagged and verified.")

## 6. Run five checks against the deployed agent

You can now invoke the deployed agent through `InvokeAgentRuntime`. Each request uses a separate session ID.

The tools return structured `grounding_result` and `command_result` values. The tests check these values to verify retrieval and Neo4j decisions. They also check selected response text to confirm that the model used the retrieved facts.

In [ ]:
from neo4j import GraphDatabase

from workshop.contracts import MAX_GUESTS, OVER_LIMIT_GUESTS
from workshop.fixtures import (
    HERO_ADDRESS,
    HERO_NAME,
    HERO_RATING,
    HERO_SOURCE,
    load_manifest,
)
from workshop.hybrid_retrieval import Neo4jConfig

# Asks for the address explicitly, because the assertion below compares the
# exact recorded string. A question that never asks for the address cannot
# check that the answer carries it.
HERO_QUESTION = (
    f"What is the full street address and guest rating of {HERO_NAME}? "
    "Quote the address exactly as it is recorded."
)
AVAILABILITY_QUESTION = f"Does {HERO_NAME} guarantee room availability next weekend?"

# A hotel that is deliberately not in the graph. Paired with the hero hotel it
# separates "the agent refused" from "retrieval returned nothing", which look
# identical from outside if only one of the two is ever tested.
ABSENT_HOTEL_ID = "00000000-0000-4000-8000-000000000000"

# One caller-created UUID, reused for every delivery of the same reservation
# request. It is the idempotency key and the correlation identifier across the
# Runtime, the command, and the CloudWatch log lines for both.
REQUEST_ID = str(uuid.uuid4())

# Relative to today, never a hardcoded date. A fixed future date rots into the
# past and silently flips a passing check-in into a failing one.
CHECK_IN = (date.today() + timedelta(days=30)).isoformat()
CHECK_OUT = (date.today() + timedelta(days=32)).isoformat()

RESERVATION_QUERY = (
    "MATCH (r:ReservationRequest {request_id: $rid})-[:FOR_HOTEL]->(h:Hotel) "
    "RETURN r.status AS status, r.guests AS guests, h.hotel_id AS hotel_id, "
    "h.name AS hotel_name, toString(r.created_at) AS created_at"
)


def ask(prompt, runtime_arn, request_id=None, session_id=None):
    """Invoke the deployed Runtime once and print what came back.

    `runtime_arn` is a parameter rather than a closure over the module-level
    RUNTIME_ARN, so every call site shows which Runtime is being invoked. A
    helper that silently picks up whatever ARN happens to be bound is the kind
    of thing that keeps working against a Runtime you thought you tore down.
    """
    payload = {"prompt": prompt}
    if request_id is not None:
        payload["request_id"] = request_id

    client = boto3.client("bedrock-agentcore", region_name=REGION)
    response = client.invoke_agent_runtime(
        agentRuntimeArn=runtime_arn,
        runtimeSessionId=session_id or str(uuid.uuid4()),
        payload=json.dumps(payload).encode("utf-8"),
        qualifier="DEFAULT",
    )
    result = json.loads(response["response"].read())

    print(f"Q: {prompt}\n")
    print(f"A: {result.get('response')}\n")
    print(f"tools used:       {result.get('tools_used') or 'none'}")
    print(f"grounding result: {result.get('grounding_result') or 'none'}")
    print(f"command result:   {result.get('command_result') or 'none'}")
    return result


def reservation_rows(request_id):
    """Read back what the deployed agent actually wrote to the graph.

    The response text is the model's account of what happened. This is the
    graph's, and only the second one is authoritative.
    """
    config = Neo4jConfig.from_environment()
    driver = GraphDatabase.driver(config.uri, auth=(config.username, config.password))
    try:
        with driver.session(database=config.database) as session:
            return [record.data() for record in session.run(RESERVATION_QUERY, rid=request_id)]
    finally:
        driver.close()


if DEPLOY_READY:
    HERO_ID = load_manifest().hotels[HERO_SOURCE]
    print(f"Hero hotel_id:  {HERO_ID}")
    print(f"Request ID:     {REQUEST_ID}")
    print(f"Stay:           {CHECK_IN} to {CHECK_OUT}")

### Test 1: Verify that the agent retrieves hotel details

The later tests expect the agent to decline or reject requests. Those results are useful only after successful retrieval confirms that the graph, index, and credentials work. This test checks that the Runtime returns the fixture hotel's exact recorded address and rating.

The expected values come from `workshop.fixtures`. The test stays aligned when the fixture hotel changes.

In [ ]:
if not DEPLOY_READY:
    print("Skipping: nothing was deployed.")
else:
    hero_result = ask(HERO_QUESTION, RUNTIME_ARN)
    grounding = hero_result.get("grounding_result") or {}
    top = grounding.get("top_result") or {}

    assert grounding.get("answerable") is True, grounding
    assert HERO_ID in (grounding.get("hotel_ids") or []), grounding

    # Exact values, never a substring and never a non-empty check. "Cairo"
    # appears in several documents and `is not None` passes on anything, so
    # either one would go green against a graph that had lost the hero hotel.
    assert top.get("hotel_id") == HERO_ID, top
    assert top.get("hotel_name") == HERO_NAME, top
    assert top.get("address") == HERO_ADDRESS, top
    assert top.get("guest_rating") == HERO_RATING, top

    # The tool returned the right facts. This asserts the model also answered
    # with them, which covers the whole path rather than just the retriever.
    assert HERO_ADDRESS in (hero_result.get("response") or ""), hero_result.get("response")
    print(f"\nPASS: returned the recorded address {HERO_ADDRESS} and rating {HERO_RATING}.")

### Test 2: Reject a reservation for an unknown hotel ID

This test requests a reservation with a hotel ID that is absent from the graph and the retrieved context. It checks that the request is not accepted and that Neo4j contains no reservation for that ID. The model does not need to call the reservation command.

In [ ]:
if not DEPLOY_READY:
    print("Skipping: nothing was deployed.")
else:
    absent_request_id = str(uuid.uuid4())
    absent = ask(
        f"Create a reservation request at hotel {ABSENT_HOTEL_ID} from {CHECK_IN} "
        f"to {CHECK_OUT} for 2 guests.",
        RUNTIME_ARN,
        request_id=absent_request_id,
    )
    absent_command = absent.get("command_result") or {}

    assert absent_command.get("status") != "accepted", absent_command
    assert reservation_rows(absent_request_id) == [], "wrote a request for a hotel that does not exist"
    assert ABSENT_HOTEL_ID not in ((absent.get("grounding_result") or {}).get("hotel_ids") or [])
    print("\nPASS: refused an ungrounded hotel_id, nothing written.")

### Test 3: Decline a question about live room availability

The graph contains hotel information but no live room inventory. The grounding tool returns `answerable: false` with `missing_fact: live_room_availability`. The result also includes the hero hotel's exact address. That address confirms that retrieval succeeded before the agent declined to answer.

In [ ]:
if not DEPLOY_READY:
    print("Skipping: nothing was deployed.")
else:
    availability_result = ask(AVAILABILITY_QUESTION, RUNTIME_ARN)
    grounding = availability_result.get("grounding_result") or {}

    assert grounding.get("answerable") is False, grounding
    assert grounding.get("missing_fact") == "live_room_availability", grounding

    # The control that makes the abstention mean something. Without it this
    # cell passes just as happily against an index that returns nothing.
    top = grounding.get("top_result") or {}
    assert top.get("address") == HERO_ADDRESS, top
    print("\nPASS: abstained on availability while retrieval was demonstrably live.")

### Test 4: Reject an over-limit reservation before Neo4j writes it

Neo4j enforces the maximum-guests rule in the transaction that writes the reservation. The prompt tells the agent to retrieve the `hotel_id` for the supplied hotel name before it calls the command. The test checks that ID, the rejection reason, and the absence of a reservation node.

In [ ]:
if not DEPLOY_READY:
    print("Skipping: nothing was deployed.")
else:
    rejected = ask(
        f"Create a reservation request at the hotel named {HERO_NAME} from "
        f"{CHECK_IN} to {CHECK_OUT} for {OVER_LIMIT_GUESTS} guests.",
        RUNTIME_ARN,
        request_id=REQUEST_ID,
    )
    command = rejected.get("command_result") or {}

    assert command.get("status") == "rejected", command
    assert command.get("reason_code") == "max_guests_exceeded", command
    # Grounded from search, not from the prompt, which never contained an ID.
    assert command.get("hotel_id") == HERO_ID, command
    assert reservation_rows(REQUEST_ID) == [], "an over-limit request was written"
    print(f"\nPASS: rejected with reason_code={command.get('reason_code')}, no node written.")

### Test 5: Write one valid reservation request and handle a retry

A request within the guest limit creates one `ReservationRequest` linked to the fixture hotel. Sending the same request again with the same `request_id` returns `duplicate=true` with the original `created_at`. The test confirms that the retry does not create a second node.

In [ ]:
if not DEPLOY_READY:
    print("Skipping: nothing was deployed.")
else:
    reservation_prompt = (
        f"Create a reservation request at the hotel named {HERO_NAME} from "
        f"{CHECK_IN} to {CHECK_OUT} for {MAX_GUESTS} guests."
    )
    accepted = ask(reservation_prompt, RUNTIME_ARN, request_id=REQUEST_ID)
    command = accepted.get("command_result") or {}
    assert command.get("status") == "accepted", command
    assert command.get("hotel_id") == HERO_ID, command

    rows = reservation_rows(REQUEST_ID)
    assert len(rows) == 1, rows
    assert rows[0]["hotel_id"] == HERO_ID, rows
    assert rows[0]["guests"] == MAX_GUESTS, rows
    print(f"\nGraph says: {json.dumps(rows[0], indent=2)}")

    # A separate session ID, because a replay from a different caller is the
    # case that matters. Sharing the session would let conversation history,
    # rather than the idempotency key, explain a correct answer.
    replay = ask(reservation_prompt, RUNTIME_ARN, request_id=REQUEST_ID)
    replay_command = replay.get("command_result") or {}
    assert replay_command.get("duplicate") is True, replay_command
    assert len(reservation_rows(REQUEST_ID)) == 1, "replay created a second node"
    print("\nPASS: one node written, replay returned duplicate=true, still one node.")

## 7. Read recent logs from the Runtime

Each successful invocation logs a start line and a completion line. The completion line includes the tools used and command status. Run the next cell to read recent entries from the Runtime's CloudWatch log group without leaving the notebook. Use the caller-provided `request_id` to correlate reservation requests.

In [ ]:
if not DEPLOY_READY or not RUNTIME_ID:
    print("Skipping logs: nothing was deployed.")
else:
    from datetime import datetime, timezone

    logs_client = boto3.client("logs", region_name=REGION)
    log_group = f"/aws/bedrock-agentcore/runtimes/{RUNTIME_ID}-DEFAULT"
    start_time = int(
        (datetime.now(timezone.utc) - timedelta(hours=2)).timestamp() * 1000
    )
    try:
        response = logs_client.filter_log_events(
            logGroupName=log_group,
            startTime=start_time,
            limit=100,
        )
    except logs_client.exceptions.ResourceNotFoundException:
        print(f"No log group found yet: {log_group}")
    else:
        events = response.get("events", [])
        print(f"Recent events from {log_group}: {len(events)}")
        for event in events:
            timestamp = datetime.fromtimestamp(
                event["timestamp"] / 1000,
                tz=timezone.utc,
            ).isoformat()
            print(f"{timestamp}  {event['message'].rstrip()}")

## Review the deployed agent architecture

```
InvokeAgentRuntime
        |
        v
+---------------------------+
|  AgentCore Runtime        |
|  GraphRagBookingAgent     |
|                           |
|  booking_agent.py         |
|   +- search_hotel_knowledge  --> Neo4j hybrid retrieval
|   +- create_reservation      --> Neo4j write, rule enforced in-transaction
|   +- BedrockModel            --> Claude on Amazon Bedrock
+---------------------------+
```

AgentCore manages the container and passes the Neo4j credentials at launch. Authorized callers invoke the agent with `bedrock-agentcore:InvokeAgentRuntime` instead of running its Python environment locally. The deployment reuses the Module 3.1 retrieval code, grounding instructions, and database-enforced reservation command. It also exposes the command as an agent tool and adds Runtime request handling.

The smoke tests verified three results:

- **Retrieval:** The hotel-details test returned the exact recorded address. This confirms that the graph and index are available.
- **Safe answers:** The availability tool result returned `answerable: false` because the graph has no live inventory.
- **Reservation rule:** Neo4j rejected the 15-guest request inside the write transaction. The graph contains no reservation node for that request.

:::alert{type="info" header="Clean Up Resources When You Finish"}
Module 6 does not use this Runtime. The workshop does not delete the Module 5 resources automatically. Find them by their `WorkshopResource` tag and remove the Runtime, ECR repository, CodeBuild project, and execution role.
:::